In [1]:
import pandas as pd
import os

DATA_PATH = "../data/dataset_sales.csv"

sales = pd.read_csv(DATA_PATH)

print("Filas:", len(sales))
print("Columnas:", sales.columns.tolist())
sales.head()

Filas: 110197
Columnas: ['order_id', 'customer_id', 'product_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'avg_review_score', 'num_reviews', 'last_review_timestamp']


,order_id,customer_id,product_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,...,product_height_cm,product_width_cm,product_category_name_english,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,avg_review_score,num_reviews,last_review_timestamp
0,0132451f29a10b66a5cf1bacc85f9afe,1c5b37d20011c637ad1a5b6d423c7483,42a2bd596fda1baef5719cb74f73030c,delivered,2017-11-08T17:08:56.000Z,2017-11-09T17:08:27.000Z,2017-11-10T18:48:59.000Z,2017-11-17T18:58:42.000Z,2017-12-07T00:00:00.000Z,1,...,2.0,35.0,bed_bath_table,aaff146bd210808c7d004a05966472a7,24431,sao goncalo,RJ,5.0,1.0,2017-11-20T10:22:14.000Z
1,01e662008f03009eb9b97f9333fcd6b4,bd13ebe5d7f8f2923430da33ffcf40ec,460a66fcc404a3d7306d5f50fcb2d18a,delivered,2018-06-30T10:04:41.000Z,2018-06-30T10:15:12.000Z,2018-07-03T08:38:00.000Z,2018-07-14T13:56:47.000Z,2018-07-20T00:00:00.000Z,1,...,16.0,17.0,housewares,3238c58997e72431e9e011b0cf7a8749,12403,pindamonhangaba,SP,3.0,1.0,2018-07-15T20:38:06.000Z
2,036091098bdc85e780fc3594006276b7,92dd835c0a01f474014b0a888b5bef93,13b4ff901d43edec666e3253b14b3a8f,delivered,2018-07-19T12:59:52.000Z,2018-07-20T14:50:20.000Z,2018-07-23T13:25:00.000Z,2018-07-27T18:28:23.000Z,2018-08-02T00:00:00.000Z,1,...,10.0,16.0,bed_bath_table,91049771b745aa43500e4c82ed6a2241,2276,sao paulo,SP,5.0,1.0,2018-07-28T21:38:11.000Z
3,036091098bdc85e780fc3594006276b7,92dd835c0a01f474014b0a888b5bef93,13b4ff901d43edec666e3253b14b3a8f,delivered,2018-07-19T12:59:52.000Z,2018-07-20T14:50:20.000Z,2018-07-23T13:25:00.000Z,2018-07-27T18:28:23.000Z,2018-08-02T00:00:00.000Z,2,...,10.0,16.0,bed_bath_table,91049771b745aa43500e4c82ed6a2241,2276,sao paulo,SP,5.0,1.0,2018-07-28T21:38:11.000Z
4,0452092bf08cf43ddab32ebec9e75d2d,bc2723356fa3f373b455c0a3112dc495,290ada89b05e1dca2a4fe42f1cfc012e,delivered,2017-02-21T10:53:34.000Z,2017-02-22T03:15:16.000Z,2017-02-22T09:34:31.000Z,2017-03-16T10:41:44.000Z,2017-03-30T00:00:00.000Z,1,...,5.0,12.0,fixed_telephony,4af923ab1facb0d2b0511746cffa138d,50730,recife,PE,5.0,1.0,2017-03-22T22:22:41.000Z


In [2]:
# Copia de trabajo
rec_df = sales.copy()

# Nos quedamos con pedidos entregados y datos mínimos válidos
rec_df = rec_df[
    (rec_df["order_status"] == "delivered") &
    (rec_df["customer_unique_id"].notna()) &
    (rec_df["product_id"].notna()) &
    (rec_df["product_category_name_english"].notna())
].copy()

# Rellenar reviews faltantes con media global
global_review_mean = rec_df["avg_review_score"].mean()
rec_df["avg_review_score"] = rec_df["avg_review_score"].fillna(global_review_mean)

# Asegurar numéricos
rec_df["price"] = pd.to_numeric(rec_df["price"], errors="coerce")
rec_df["freight_value"] = pd.to_numeric(rec_df["freight_value"], errors="coerce")

print("Filas para recomendador:", len(rec_df))
print("Clientes únicos:", rec_df["customer_unique_id"].nunique())
print("Productos únicos:", rec_df["product_id"].nunique())
print("Categorías:", rec_df["product_category_name_english"].nunique())

rec_df.head()

Filas para recomendador: 108638
Clientes únicos: 92079
Productos únicos: 31621
Categorías: 71


,order_id,customer_id,product_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,...,product_height_cm,product_width_cm,product_category_name_english,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,avg_review_score,num_reviews,last_review_timestamp
0,0132451f29a10b66a5cf1bacc85f9afe,1c5b37d20011c637ad1a5b6d423c7483,42a2bd596fda1baef5719cb74f73030c,delivered,2017-11-08T17:08:56.000Z,2017-11-09T17:08:27.000Z,2017-11-10T18:48:59.000Z,2017-11-17T18:58:42.000Z,2017-12-07T00:00:00.000Z,1,...,2.0,35.0,bed_bath_table,aaff146bd210808c7d004a05966472a7,24431,sao goncalo,RJ,5.0,1.0,2017-11-20T10:22:14.000Z
1,01e662008f03009eb9b97f9333fcd6b4,bd13ebe5d7f8f2923430da33ffcf40ec,460a66fcc404a3d7306d5f50fcb2d18a,delivered,2018-06-30T10:04:41.000Z,2018-06-30T10:15:12.000Z,2018-07-03T08:38:00.000Z,2018-07-14T13:56:47.000Z,2018-07-20T00:00:00.000Z,1,...,16.0,17.0,housewares,3238c58997e72431e9e011b0cf7a8749,12403,pindamonhangaba,SP,3.0,1.0,2018-07-15T20:38:06.000Z
2,036091098bdc85e780fc3594006276b7,92dd835c0a01f474014b0a888b5bef93,13b4ff901d43edec666e3253b14b3a8f,delivered,2018-07-19T12:59:52.000Z,2018-07-20T14:50:20.000Z,2018-07-23T13:25:00.000Z,2018-07-27T18:28:23.000Z,2018-08-02T00:00:00.000Z,1,...,10.0,16.0,bed_bath_table,91049771b745aa43500e4c82ed6a2241,2276,sao paulo,SP,5.0,1.0,2018-07-28T21:38:11.000Z
3,036091098bdc85e780fc3594006276b7,92dd835c0a01f474014b0a888b5bef93,13b4ff901d43edec666e3253b14b3a8f,delivered,2018-07-19T12:59:52.000Z,2018-07-20T14:50:20.000Z,2018-07-23T13:25:00.000Z,2018-07-27T18:28:23.000Z,2018-08-02T00:00:00.000Z,2,...,10.0,16.0,bed_bath_table,91049771b745aa43500e4c82ed6a2241,2276,sao paulo,SP,5.0,1.0,2018-07-28T21:38:11.000Z
4,0452092bf08cf43ddab32ebec9e75d2d,bc2723356fa3f373b455c0a3112dc495,290ada89b05e1dca2a4fe42f1cfc012e,delivered,2017-02-21T10:53:34.000Z,2017-02-22T03:15:16.000Z,2017-02-22T09:34:31.000Z,2017-03-16T10:41:44.000Z,2017-03-30T00:00:00.000Z,1,...,5.0,12.0,fixed_telephony,4af923ab1facb0d2b0511746cffa138d,50730,recife,PE,5.0,1.0,2017-03-22T22:22:41.000Z


In [3]:
# Ranking de productos por popularidad y valoración media
product_stats = (
    rec_df
    .groupby(["product_id", "product_category_name_english"], as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        avg_rating=("avg_review_score", "mean"),
        avg_price=("price", "mean")
    )
)

# Filtrar productos con mínimo de ventas para evitar recomendar productos anecdóticos
product_stats = product_stats[product_stats["total_orders"] >= 3].copy()

# Normalizar métricas para crear score
product_stats["orders_norm"] = product_stats["total_orders"] / product_stats["total_orders"].max()
product_stats["rating_norm"] = product_stats["avg_rating"] / 5

# Score híbrido: popularidad + valoración
product_stats["recommendation_score"] = (
    0.7 * product_stats["orders_norm"] +
    0.3 * product_stats["rating_norm"]
)

product_stats = product_stats.sort_values(
    "recommendation_score",
    ascending=False
).reset_index(drop=True)

print("Productos recomendables:", len(product_stats))
product_stats.head(10)

Productos recomendables: 7678


,product_id,product_category_name_english,total_orders,avg_rating,avg_price,orders_norm,rating_norm,recommendation_score
0,99a4788cb24856965c36a24e339b6058,bed_bath_table,456,3.929085,88.154423,1.000000,0.785817,0.935745
1,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,425,4.048563,71.354423,0.932018,0.809713,0.895326
2,422879e10f46682990de24d770e7f83d,garden_tools,352,3.962984,54.911612,0.771930,0.792597,0.778130
3,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,313,4.283640,137.411325,0.686404,0.856728,0.737501
4,389d119b48cf3043d311335e499d9c6b,garden_tools,309,4.133549,54.709718,0.677632,0.826710,0.722355
5,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,304,4.199901,116.681090,0.666667,0.839980,0.718661
6,368c6c730842d78016ad823897a372db,garden_tools,291,3.924620,54.270103,0.638158,0.784924,0.682188
7,53759a2ecddad2bb87a079a1f1519f73,garden_tools,287,3.865063,54.657373,0.629386,0.773013,0.672474
8,154e7e31ebfa092203795c972e5804a6,health_beauty,262,4.379869,22.530146,0.574561,0.875974,0.664985
9,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,253,4.220898,149.936765,0.554825,0.844180,0.641631


In [4]:
# Conteo de compras por cliente y categoría
customer_category_counts = (
    rec_df
    .groupby(["customer_unique_id", "product_category_name_english"], as_index=False)
    .agg(
        category_orders=("order_id", "nunique"),
        category_spend=("price", "sum"),
        avg_category_rating=("avg_review_score", "mean")
    )
)

# Ordenar para quedarnos con la categoría más comprada por cliente
customer_favorite_category = (
    customer_category_counts
    .sort_values(
        ["customer_unique_id", "category_orders", "category_spend"],
        ascending=[True, False, False]
    )
    .drop_duplicates("customer_unique_id")
    .reset_index(drop=True)
)

customer_favorite_category = customer_favorite_category.rename(
    columns={"product_category_name_english": "favorite_category"}
)

print("Clientes con categoría favorita:", len(customer_favorite_category))
customer_favorite_category.head()

Clientes con categoría favorita: 92079


,customer_unique_id,favorite_category,category_orders,category_spend,avg_category_rating
0,0000366f3b9a7992bf8c76cfdf3221e2,bed_bath_table,1,129.90,5.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,health_beauty,1,18.90,4.0
2,0000f46a3911fa3c0805444483337064,stationery,1,69.00,3.0
3,0000f6ccb0745a6a4b88665a16c9f078,telephony,1,25.99,4.0
4,0004aac84e0df4da2b147fca70cf8255,telephony,1,180.00,5.0


In [5]:
# Productos ya comprados por cada cliente
customer_purchased_products = (
    rec_df
    .groupby("customer_unique_id")["product_id"]
    .apply(set)
    .to_dict()
)

# Ranking de productos por categoría
category_rankings = {
    category: group.sort_values("recommendation_score", ascending=False).to_dict("records")
    for category, group in product_stats.groupby("product_category_name_english")
}

# Ranking global por si una categoría no tiene suficientes productos
global_ranking = product_stats.sort_values("recommendation_score", ascending=False).to_dict("records")

print("Rankings por categoría:", len(category_rankings))
print("Ranking global:", len(global_ranking))

Rankings por categoría: 69
Ranking global: 7678


In [6]:
def recommend_for_customer(customer_id, favorite_category, top_n=5):
    purchased = customer_purchased_products.get(customer_id, set())
    recommendations = []

    # 1. Intentar recomendar dentro de la categoría favorita
    category_products = category_rankings.get(favorite_category, [])

    for product in category_products:
        if product["product_id"] not in purchased:
            recommendations.append({
                "customer_unique_id": customer_id,
                "recommended_product_id": product["product_id"],
                "product_category_name_english": product["product_category_name_english"],
                "score": product["recommendation_score"],
                "avg_rating": product["avg_rating"],
                "total_orders": product["total_orders"],
                "avg_price": product["avg_price"],
                "reason": f"Popular product in customer's favorite category: {favorite_category}"
            })

        if len(recommendations) >= top_n:
            break

    # 2. Si no hay suficientes, completar con ranking global
    if len(recommendations) < top_n:
        for product in global_ranking:
            if product["product_id"] not in purchased:
                recommendations.append({
                    "customer_unique_id": customer_id,
                    "recommended_product_id": product["product_id"],
                    "product_category_name_english": product["product_category_name_english"],
                    "score": product["recommendation_score"],
                    "avg_rating": product["avg_rating"],
                    "total_orders": product["total_orders"],
                    "avg_price": product["avg_price"],
                    "reason": "Global popular product fallback"
                })

            if len(recommendations) >= top_n:
                break

    return recommendations

In [7]:
example_customer = customer_favorite_category.iloc[0]["customer_unique_id"]
example_category = customer_favorite_category.iloc[0]["favorite_category"]

example_recs = recommend_for_customer(example_customer, example_category, top_n=5)

pd.DataFrame(example_recs)

,customer_unique_id,recommended_product_id,product_category_name_english,score,avg_rating,total_orders,avg_price,reason
0,0000366f3b9a7992bf8c76cfdf3221e2,99a4788cb24856965c36a24e339b6058,bed_bath_table,0.935745,3.929085,456,88.154423,Popular product in customer's favorite categor...
1,0000366f3b9a7992bf8c76cfdf3221e2,f1c7f353075ce59d8a6f3cf58f419c9c,bed_bath_table,0.491147,4.373650,149,194.721307,Popular product in customer's favorite categor...
2,0000366f3b9a7992bf8c76cfdf3221e2,06edb72f1e0c64b14c5b79353f7abea3,bed_bath_table,0.439099,4.043459,128,40.798571,Popular product in customer's favorite categor...
3,0000366f3b9a7992bf8c76cfdf3221e2,ec2d43cc59763ec91694573b31f1c29a,bed_bath_table,0.422231,4.069345,116,45.891374,Popular product in customer's favorite categor...
4,0000366f3b9a7992bf8c76cfdf3221e2,777d2e438a1b645f3aec9bd57e92672c,bed_bath_table,0.383145,4.108696,89,69.869565,Popular product in customer's favorite categor...


In [8]:
all_recommendations = []

for row in customer_favorite_category.itertuples(index=False):
    customer_id = row.customer_unique_id
    favorite_category = row.favorite_category
    
    recs = recommend_for_customer(
        customer_id=customer_id,
        favorite_category=favorite_category,
        top_n=5
    )
    
    all_recommendations.extend(recs)

customer_recommendations = pd.DataFrame(all_recommendations)

print("Recomendaciones generadas:", len(customer_recommendations))
print("Clientes con recomendaciones:", customer_recommendations["customer_unique_id"].nunique())

customer_recommendations.head()

Recomendaciones generadas: 460395
Clientes con recomendaciones: 92079


,customer_unique_id,recommended_product_id,product_category_name_english,score,avg_rating,total_orders,avg_price,reason
0,0000366f3b9a7992bf8c76cfdf3221e2,99a4788cb24856965c36a24e339b6058,bed_bath_table,0.935745,3.929085,456,88.154423,Popular product in customer's favorite categor...
1,0000366f3b9a7992bf8c76cfdf3221e2,f1c7f353075ce59d8a6f3cf58f419c9c,bed_bath_table,0.491147,4.373650,149,194.721307,Popular product in customer's favorite categor...
2,0000366f3b9a7992bf8c76cfdf3221e2,06edb72f1e0c64b14c5b79353f7abea3,bed_bath_table,0.439099,4.043459,128,40.798571,Popular product in customer's favorite categor...
3,0000366f3b9a7992bf8c76cfdf3221e2,ec2d43cc59763ec91694573b31f1c29a,bed_bath_table,0.422231,4.069345,116,45.891374,Popular product in customer's favorite categor...
4,0000366f3b9a7992bf8c76cfdf3221e2,777d2e438a1b645f3aec9bd57e92672c,bed_bath_table,0.383145,4.108696,89,69.869565,Popular product in customer's favorite categor...


In [9]:
os.makedirs("../outputs/recommendations", exist_ok=True)

customer_recommendations.to_csv(
    "../outputs/recommendations/customer_recommendations.csv",
    index=False,
    encoding="utf-8"
)

product_stats.to_csv(
    "../outputs/recommendations/global_product_recommendations.csv",
    index=False,
    encoding="utf-8"
)

customer_favorite_category.to_csv(
    "../outputs/recommendations/customer_favorite_category.csv",
    index=False,
    encoding="utf-8"
)

print("Archivos guardados:")
print("../outputs/recommendations/customer_recommendations.csv")
print("../outputs/recommendations/global_product_recommendations.csv")
print("../outputs/recommendations/customer_favorite_category.csv")

Archivos guardados:
../outputs/recommendations/customer_recommendations.csv
../outputs/recommendations/global_product_recommendations.csv
../outputs/recommendations/customer_favorite_category.csv


In [10]:
check = pd.read_csv("../outputs/recommendations/customer_recommendations.csv")

print("Filas:", len(check))
print("Clientes:", check["customer_unique_id"].nunique())
check.head()

Filas: 460395
Clientes: 92079


,customer_unique_id,recommended_product_id,product_category_name_english,score,avg_rating,total_orders,avg_price,reason
0,0000366f3b9a7992bf8c76cfdf3221e2,99a4788cb24856965c36a24e339b6058,bed_bath_table,0.935745,3.929085,456,88.154423,Popular product in customer's favorite categor...
1,0000366f3b9a7992bf8c76cfdf3221e2,f1c7f353075ce59d8a6f3cf58f419c9c,bed_bath_table,0.491147,4.373650,149,194.721307,Popular product in customer's favorite categor...
2,0000366f3b9a7992bf8c76cfdf3221e2,06edb72f1e0c64b14c5b79353f7abea3,bed_bath_table,0.439099,4.043459,128,40.798571,Popular product in customer's favorite categor...
3,0000366f3b9a7992bf8c76cfdf3221e2,ec2d43cc59763ec91694573b31f1c29a,bed_bath_table,0.422231,4.069345,116,45.891374,Popular product in customer's favorite categor...
4,0000366f3b9a7992bf8c76cfdf3221e2,777d2e438a1b645f3aec9bd57e92672c,bed_bath_table,0.383145,4.108696,89,69.869565,Popular product in customer's favorite categor...
